# Lab 3: Clustering Analysis Using K-Means and K-Medoids Algorithms

**Student:** Minh Le  
**University:** University of the Cumberlands  
**Course:** 2026 Fall - Advanced Big Data and Data Mining (MSCS-634-M50) - Full Term

---

This notebook applies K-Means and K-Medoids clustering to the scikit-learn Wine dataset. The analysis explores and standardizes the data, evaluates both algorithms with the Silhouette Score and Adjusted Rand Index (ARI), visualizes their clusters in two principal components, and discusses the practical tradeoffs between the methods.

## 1. Setup and reproducibility

A fixed random seed is used throughout so that the clustering results can be reproduced. The evaluation metrics are computed in the full 13-dimensional standardized feature space; PCA is used only to display the clusters in two dimensions.

In [ ]:
%pip install numpy pandas matplotlib scikit-learn

In [ ]:
%matplotlib inline

# Import the libraries used for data preparation, clustering, evaluation, and plotting.
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, pairwise_distances, silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

# Use one seed for every randomized step so the results are reproducible.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Record package versions to make the execution environment easier to reproduce.
print(f"Python: {sys.version.split()[0]}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

## 2. Load and explore the Wine dataset

In [ ]:
# Load the built-in Wine dataset; no external download is required.
wine = load_wine()
# Keep predictors (X) separate from the known class labels (y).
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = pd.Series(wine.target, name="actual_class")

# Check the dataset dimensions, completeness, and available labels.
print(f"Dataset shape: {X.shape[0]} samples x {X.shape[1]} features")
print(f"Missing values: {int(X.isna().sum().sum())}")
print(f"Target names: {list(wine.target_names)}")
print("\nFeature names:")
print(list(X.columns))

X.head()

In [ ]:
# Count each reference class and calculate its share of the full dataset.
class_distribution = (
    y.value_counts()
    .sort_index()
    .rename(index=dict(enumerate(wine.target_names)))
    .rename_axis("wine_class")
    .to_frame("sample_count")
)
class_distribution["percentage"] = (
    100 * class_distribution["sample_count"] / len(y)
).round(2)

print("Class distribution:")
display(class_distribution)

# Transpose the summary so each feature is displayed on one readable row.
print("Descriptive statistics (transposed):")
display(X.describe().T.round(3))

The dataset contains 178 observations, 13 continuous chemical measurements, and three known wine classes. There are no missing values. The descriptive statistics show that the features have very different units and ranges, so distance-based clustering on the raw values would allow large-scale features to dominate.

## 3. Standardize the features

In [ ]:
# Standardize every feature to mean 0 and standard deviation 1.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Verify that the transformation produced the intended scale.
scale_check = pd.DataFrame(
    {
        "mean_after_scaling": X_scaled.mean(axis=0),
        "std_after_scaling": X_scaled.std(axis=0, ddof=0),
    },
    index=X.columns,
)

print("Largest absolute standardized mean:", f"{np.abs(X_scaled.mean(axis=0)).max():.3e}")
print("Standard-deviation range:", f"{X_scaled.std(axis=0).min():.3f} to {X_scaled.std(axis=0).max():.3f}")
display(scale_check.round(4))

Z-score standardization gives every feature a mean near 0 and a population standard deviation of 1. This makes Euclidean distances across the 13 features comparable.

## 4. K-Means clustering (k = 3)

In [ ]:
# Fit three clusters and use multiple initializations to reduce sensitivity to starting centroids.
kmeans = KMeans(n_clusters=3, n_init=20, random_state=RANDOM_STATE)
kmeans_labels = kmeans.fit_predict(X_scaled)

# Evaluate internal separation and agreement with the known wine classes.
kmeans_silhouette = silhouette_score(X_scaled, kmeans_labels)
kmeans_ari = adjusted_rand_score(y, kmeans_labels)

print(f"K-Means silhouette score: {kmeans_silhouette:.4f}")
print(f"K-Means adjusted Rand index: {kmeans_ari:.4f}")
print("K-Means cluster sizes:", dict(zip(*np.unique(kmeans_labels, return_counts=True))))

# A contingency table makes class-to-cluster agreement visible despite arbitrary cluster IDs.
print("\nActual class by K-Means cluster:")
display(pd.crosstab(y, kmeans_labels, rownames=["actual_class"], colnames=["cluster"]))

The Silhouette Score ranges from -1 to 1; larger values indicate that samples are more cohesive within their assigned clusters and better separated from other clusters. ARI compares the cluster assignments with the known classes while correcting for chance. ARI also ranges up to 1, where 1 indicates exact agreement; cluster label numbers themselves are arbitrary.

## 5. K-Medoids clustering (k = 3)

The following implementation uses an alternating K-Medoids procedure with multiple random starts. It assigns each sample to its nearest medoid, then replaces each medoid with the member that minimizes the total within-cluster Euclidean distance. Unlike a centroid, every medoid is an actual observation from the dataset.

In [ ]:
def fit_kmedoids(X_values, n_clusters=3, n_init=50, max_iter=100, random_state=42):
    # Fit K-Medoids by alternating assignment and within-cluster medoid updates.
    X_values = np.asarray(X_values, dtype=float)
    n_samples = X_values.shape[0]
    # Precompute all pairwise distances because the same values are reused in every iteration.
    distances = pairwise_distances(X_values, metric="euclidean")
    rng = np.random.default_rng(random_state)

    # Try several random starting medoids and retain the solution with the lowest total cost.
    best = None
    for _ in range(n_init):
        medoid_indices = np.sort(rng.choice(n_samples, size=n_clusters, replace=False))

        for iteration in range(1, max_iter + 1):
            # Assign every observation to its nearest current medoid.
            labels = np.argmin(distances[:, medoid_indices], axis=1)
            new_medoids = medoid_indices.copy()

            for cluster_id in range(n_clusters):
                members = np.flatnonzero(labels == cluster_id)
                if len(members) == 0:
                    # Recover an empty cluster with the point farthest from all current medoids.
                    non_medoids = np.setdiff1d(np.arange(n_samples), medoid_indices)
                    distance_to_nearest = distances[non_medoids][:, medoid_indices].min(axis=1)
                    new_medoids[cluster_id] = non_medoids[np.argmax(distance_to_nearest)]
                else:
                    # The new medoid is the real member minimizing total distance to its cluster.
                    within_cluster = distances[np.ix_(members, members)]
                    new_medoids[cluster_id] = members[np.argmin(within_cluster.sum(axis=1))]

            # Stop when an update no longer changes any medoid.
            if np.array_equal(new_medoids, medoid_indices):
                break
            medoid_indices = new_medoids

        labels = np.argmin(distances[:, medoid_indices], axis=1)
        # The objective is the sum of each observation's distance to its assigned medoid.
        total_cost = distances[np.arange(n_samples), medoid_indices[labels]].sum()

        if best is None or total_cost < best["cost"]:
            best = {
                "medoid_indices": medoid_indices.copy(),
                "labels": labels.copy(),
                "cost": float(total_cost),
                "iterations": iteration,
            }

    return best


# Fit the model on all standardized features using the same three-cluster target.
kmedoids = fit_kmedoids(
    X_scaled,
    n_clusters=3,
    n_init=50,
    max_iter=100,
    random_state=RANDOM_STATE,
)
kmedoids_labels = kmedoids["labels"]
medoid_indices = kmedoids["medoid_indices"]

# Calculate the same metrics used for K-Means for a fair comparison.
kmedoids_silhouette = silhouette_score(X_scaled, kmedoids_labels)
kmedoids_ari = adjusted_rand_score(y, kmedoids_labels)

print(f"K-Medoids silhouette score: {kmedoids_silhouette:.4f}")
print(f"K-Medoids adjusted Rand index: {kmedoids_ari:.4f}")
print(f"K-Medoids objective (sum of distances): {kmedoids['cost']:.4f}")
print("Medoid row indices:", medoid_indices.tolist())
print("K-Medoids cluster sizes:", dict(zip(*np.unique(kmedoids_labels, return_counts=True))))

print("\nActual class by K-Medoids cluster:")
display(pd.crosstab(y, kmedoids_labels, rownames=["actual_class"], colnames=["cluster"]))

## 6. Compare metrics

In [ ]:
# Place both algorithms and metrics in one table for direct comparison.
results = pd.DataFrame(
    {
        "Silhouette Score": [kmeans_silhouette, kmedoids_silhouette],
        "Adjusted Rand Index": [kmeans_ari, kmedoids_ari],
    },
    index=["K-Means", "K-Medoids"],
)

display(results.round(4))

# Higher values are better for both metrics.
best_silhouette = results["Silhouette Score"].idxmax()
best_ari = results["Adjusted Rand Index"].idxmax()
print(f"Better-defined clusters by silhouette: {best_silhouette}")
print(f"Closer agreement with known wine classes by ARI: {best_ari}")

## 7. Visualize the clusters with PCA

In [ ]:
# Reduce the standardized data to two components only for visualization.
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
# Project the K-Means centroids and locate the observed K-Medoids representatives.
kmeans_centers_pca = pca.transform(kmeans.cluster_centers_)
kmedoids_points_pca = X_pca[medoid_indices]

fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharex=True, sharey=True)
colors = ["#2a9d8f", "#e9c46a", "#e76f51"]

# Draw K-Means assignments in the left panel.
for cluster_id, color in enumerate(colors):
    km_mask = kmeans_labels == cluster_id
    axes[0].scatter(
        X_pca[km_mask, 0], X_pca[km_mask, 1],
        s=42, alpha=0.78, color=color, edgecolor="white", linewidth=0.35,
        label=f"Cluster {cluster_id}",
    )
axes[0].scatter(
    kmeans_centers_pca[:, 0], kmeans_centers_pca[:, 1],
    marker="X", s=260, color="#1d3557", edgecolor="white", linewidth=1.4,
    label="Centroids", zorder=5,
)
axes[0].set_title("K-Means Clusters", fontweight="bold")

# Draw K-Medoids assignments in the right panel using the same colors and axes.
for cluster_id, color in enumerate(colors):
    kmed_mask = kmedoids_labels == cluster_id
    axes[1].scatter(
        X_pca[kmed_mask, 0], X_pca[kmed_mask, 1],
        s=42, alpha=0.78, color=color, edgecolor="white", linewidth=0.35,
        label=f"Cluster {cluster_id}",
    )
axes[1].scatter(
    kmedoids_points_pca[:, 0], kmedoids_points_pca[:, 1],
    marker="P", s=260, color="#1d3557", edgecolor="white", linewidth=1.4,
    label="Medoids", zorder=5,
)
axes[1].set_title("K-Medoids Clusters", fontweight="bold")

# Label each axis with the percentage of original variance it preserves.
explained = pca.explained_variance_ratio_ * 100
for ax in axes:
    ax.set_xlabel(f"Principal Component 1 ({explained[0]:.1f}% variance)")
    ax.set_ylabel(f"Principal Component 2 ({explained[1]:.1f}% variance)")
    ax.grid(alpha=0.18)
    ax.legend(frameon=True, fontsize=9)

fig.suptitle("Wine Dataset Clustering Comparison (PCA Projection)", fontsize=15, fontweight="bold")
fig.tight_layout()
display(fig)
plt.close(fig)

print(f"Variance represented by the two-dimensional view: {explained.sum():.1f}%")

## 8. Analysis and conclusions

In [ ]:
# Positive differences mean K-Means scored higher than K-Medoids.
silhouette_gap = kmeans_silhouette - kmedoids_silhouette
ari_gap = kmeans_ari - kmedoids_ari

print(
    f"K-Means achieved a silhouette score of {kmeans_silhouette:.4f} and an ARI of {kmeans_ari:.4f}, "
    f"while K-Medoids achieved {kmedoids_silhouette:.4f} and {kmedoids_ari:.4f}, respectively."
)
print(f"Silhouette difference (K-Means - K-Medoids): {silhouette_gap:+.4f}")
print(f"ARI difference (K-Means - K-Medoids): {ari_gap:+.4f}")

### Interpretation

1. **Which method produced better-defined clusters?**  
   K-Means produced the better-defined clusters in this run because it obtained the higher Silhouette Score. It also achieved the higher ARI, meaning that its assignments more closely matched the three known wine classes.

2. **What differences are visible in cluster shape or position?**  
   The PCA projection shows the same broad three-group structure for both methods, including two comparatively separated outer groups and a more mixed central boundary. K-Means places an arithmetic centroid at the middle of each cluster and therefore favors compact, roughly spherical partitions. K-Medoids anchors each cluster to a real wine sample, so its representative points and several boundary assignments shift. The plots are only a two-dimensional projection, so the numerical metrics calculated from all 13 standardized features are the stronger basis for comparison.

3. **When is each method preferable?**  
   K-Means is preferable for larger numeric datasets when speed and compact Euclidean clusters are priorities. K-Medoids is preferable when outliers may distort means, when a real observation is needed as the cluster representative, or when a custom dissimilarity measure is more appropriate than averaging features. Its main drawback is greater computational cost because candidate medoids must be selected from the observed samples.

### Final conclusion

For this standardized Wine dataset and the fixed settings used here, K-Means is the stronger choice because it provides both better internal separation and better agreement with the reference classes. K-Medoids remains useful as a more robust and interpretable alternative, particularly when representative real observations or reduced sensitivity to extreme values matter.